
# DS2002 — Homework: Stocks ETL
## APIs + DataFrames + Databases

In this homework you will build a small end-to-end ETL pipeline:

1. **Extract** data from multiple API endpoints (more than one endpoint is required)
2. **Transform** the raw JSON into clean Pandas DataFrames
3. **Cull** (remove) columns you do not need
4. **Transform** the cleaned data (compute new features)
5. **Load** the resulting DataFrames into a SQLite database
6. **Prove** the load worked by running SQL queries against your new tables

You will use the finance API described at `financeapi.net` (base URL: `https://yfapi.net`).

This assignment is designed to feel like real systems work:
- APIs return nested JSON
- you must decide what columns matter
- you must build tables that are useful later



## Submission

Submit the **Kaggle Notebook URL**.

Your notebook must show:
- completed TODO cells
- visible outputs for key steps (DataFrames displayed)
- SQL proof queries at the end (outputs visible)



## Grading Rubric (100 points)

### A) Extract: API Calls (35 points)
You must successfully call **at least 3 endpoints**.

- 15 pts: `/v6/finance/quote` for multiple tickers
- 10 pts: `/v8/finance/chart/{ticker}` for historical prices (time series)
- 10 pts: One additional endpoint of your choice (examples below)

Acceptable “third endpoint” choices include (pick ONE):
- `/v6/finance/autocomplete`
- `/v1/finance/trending/{region}`
- `/v6/finance/quote/marketSummary`
- `/v11/finance/quoteSummary/{symbol}` (requires `modules`)

### B) Transform in Pandas (35 points)
- 10 pts: Cull columns (keep only useful fields; show the before/after column lists)
- 15 pts: At least **3 real transformations** that change meaning (not just renaming)
- 10 pts: At least **1 join/merge** between DataFrames (ex: quote table + latest close)

### C) Load to SQLite (20 points)
- 10 pts: Load at least **3 DataFrames** into SQL using `.to_sql(...)`
- 10 pts: Tables have sensible names and useful columns

### D) Proof Queries (10 points)
- 10 pts: Run at least **3 SQL queries** that prove the data is present and makes sense

### Reproducibility (penalty up to -10 points)
If the notebook does not run top-to-bottom without manual fixes, points may be deducted.



## Required Setup: API Key (Kaggle)

This API requires a key.

In Kaggle:
1. Turn **Internet ON** in notebook settings.
2. Add a Kaggle **Secret** named: `YHFINANCE_API_KEY`
3. Restart the session and run the setup cell below.

Do not paste keys into notebooks after you turn them in...make sure I can see your results and then delete the key. Treat keys like passwords.



## Part 0 — Setup (Run This)

This cell:
- imports libraries
- creates a SQLite connection
- defines `q(...)` for SQL proof queries
- defines `get_json(...)` for consistent API requests

If your key is missing, this cell will tell you what to do.


In [ ]:

import os
import time
import sqlite3
import requests
import pandas as pd

# SQLite database (in-memory is fine for homework)
conn = sqlite3.connect(":memory:")

def q(sql: str) -> pd.DataFrame:
    return pd.read_sql_query(sql, conn)

BASE_URL = "https://yfapi.net"
API_KEY = os.getenv("YHFINANCE_API_KEY")

if not API_KEY:
    print("No API key found.")
    print("Kaggle → Add-ons/Settings → Secrets → add YHFINANCE_API_KEY")
    print("Then restart the session and re-run this cell.")
else:
    print("API key loaded (not printed).")

def get_json(endpoint: str, params: dict | None = None, pause_s: float = 0.25) -> dict:
    """Call the Finance API endpoint and return parsed JSON."""
    if not API_KEY:
        raise RuntimeError("Missing API key. Set YHFINANCE_API_KEY as a Kaggle Secret.")

    url = BASE_URL + endpoint
    headers = {
        "accept": "application/json",
        "X-API-KEY": API_KEY
    }
    resp = requests.get(url, headers=headers, params=params, timeout=30)
    time.sleep(pause_s)  # helps avoid rate limiting

    if resp.status_code != 200:
        raise RuntimeError(
            f"Request failed ({resp.status_code})\n"
            f"URL: {resp.url}\n"
            f"Body (first 500 chars): {resp.text[:500]}"
        )
    return resp.json()

print("Setup ready.")



## Part 1 — Choose Your Tickers (TODO)

Pick **3–6** stock tickers.

Guidance:
- choose large, well-known companies
- avoid very new or obscure tickers

Example set (use your own if you want): `AAPL, MSFT, AMZN, TSLA, NVDA`


In [ ]:

# TODO: choose 3–6 tickers
tickers = ["AAPL", "MSFT", "TSLA"]

tickers



## Part 2 — Extract #1: Quote Data for Multiple Tickers (TODO)

Endpoint: `GET /v6/finance/quote`

This endpoint can take multiple symbols at once using a comma-separated list.

Your job:
1. Build the comma-separated symbol list
2. Call the endpoint
3. Convert the results into a DataFrame `df_quote`
4. Display `df_quote.head()`

Hint:
- results are commonly nested under: `quoteResponse -> result`


In [ ]:

# TODO
# symbols = ",".join(tickers)
# HINT raw_quote = get_json("/v6/finance/quote", params={"symbols": symbols, "region": "US", "lang": "en"})

# df_quote.head()



## Part 3 — Cull Columns (TODO)

API responses contain many fields.
Your job is to keep only what your pipeline needs.

Minimum columns to keep (you may keep more if you justify them):
- `symbol`
- `shortName` (or `longName`)
- `regularMarketPrice`
- `regularMarketChange`
- `regularMarketChangePercent`
- `regularMarketVolume`
- `marketCap`
- `currency`

Your deliverable:
1. Print the original column count and column list
2. Create `df_quote_clean`
3. Print the cleaned column count and column list
4. Display `df_quote_clean.head()`


In [ ]:

# TODO


# df_quote_clean.head()



## Part 4 — Transform #1: Make the Quote Table More Useful (TODO)

Add these columns to `df_quote_clean`:

1. `marketCap_B` = marketCap expressed in **billions**
2. `price_rounded` = regularMarketPrice rounded to 2 decimals
3. `change_pct_rounded` = regularMarketChangePercent rounded to 2 decimals

Then:
- sort by `marketCap_B` descending
- display the sorted DataFrame

This is a common “transform” step in ETL: making fields more usable for humans and downstream analysis.


In [ ]:

# TODO
#HINT df_quote_clean["marketCap_B"] = df_quote_clean["marketCap"] / 1_000_000_000

# df_quote_clean.sort_values("marketCap_B", ascending=False)



## Part 5 — Extract #2: Historical Prices for One Ticker (TODO)

Endpoint: `GET /v8/finance/chart/{ticker}`

You will choose ONE ticker from your list as the focus ticker.
You will build a daily price table from the chart endpoint.

Your deliverable:
1. Set `FOCUS_TICKER`
2. Call the chart endpoint for a range like `"1mo"` or `"3mo"` and interval `"1d"`
3. Build `df_prices` with columns:
   - `symbol`
   - `date` (as datetime)
   - `close`
4. Drop rows where close is missing
5. Display `df_prices.head()` and `df_prices.tail()`

Hints:
- timestamps are often in seconds (use `pd.to_datetime(..., unit="s")`)
- close values are usually nested under indicators -> quote -> close


In [ ]:

# TODO


# df_prices.head()



## Part 6 — Transform #2: Compute Returns and Rolling Metrics (TODO)

Time series are valuable because we can compute signals.

Add these columns to `df_prices`:
1. `daily_return` = percent change of close from the previous day
2. `rolling_7d_avg_close` = 7-day rolling mean of close
3. `rolling_7d_avg_return` = 7-day rolling mean of daily_return

Deliverable:
- show the last 10 rows

Hints:
- sort by date first
- use `pct_change()`
- use `rolling(7).mean()`


In [ ]:

# TODO
#

# df_prices.tail(10)



## Part 7 — Extract #3: A Third Endpoint (Pick One) (TODO)

You must call one additional endpoint beyond quote + chart.

Pick ONE option below and build a DataFrame called `df_extra`:

Option A: Autocomplete
- endpoint: `/v6/finance/autocomplete`
- params might include: `query`, `region`

Option B: Trending
- endpoint: `/v1/finance/trending/{region}`
- example region: `US`

Option C: Market Summary
- endpoint: `/v6/finance/quote/marketSummary`
- params may include: `region` and/or `lang`

Option D: Quote Summary
- endpoint: `/v11/finance/quoteSummary/{symbol}`
- requires a `modules` parameter (request only a small set)

Deliverable:
1. Make the call
2. Convert to DataFrame `df_extra`
3. Cull columns to a small meaningful set
4. Display `df_extra.head()`


In [ ]:

# TODO
# Example (Autocomplete):
#

# df_extra.head()



## Part 8 — Transform #3: Join DataFrames (TODO)

ETL pipelines often combine data sources.

Goal:
- Take the latest close from `df_prices`
- Merge it into `df_quote_clean` on `symbol`
- Compute a new column: `price_minus_latest_close`

Deliverable:
1. Create `df_latest` (1 row) with: symbol, latest_date, latest_close
2. Merge into a new DataFrame `df_quote_enriched`
3. Add `price_minus_latest_close`
4. Display `df_quote_enriched`


In [ ]:

# TODO
#
# df_quote_enriched



## Part 9 — Load: Write DataFrames Into SQLite (TODO)

You must load at least 3 tables:
- `quotes`
- `prices`
- `extra`

Use:
- `to_sql(..., if_exists="replace", index=False)`

Deliverable:
- list the SQLite tables after writing


In [ ]:

# TODO
# df_quote_enriched.to_sql("quotes", conn, if_exists="replace", index=False)
# df_prices.to_sql("prices", conn, if_exists="replace", index=False)
# df_extra.to_sql("extra", conn, if_exists="replace", index=False)

q("SELECT name FROM sqlite_master WHERE type='table' ORDER BY name;")



## Part 10 — Proof: SQL Queries (TODO)

Run at least 3 SQL queries to prove your load worked.
Use the `q(...)` helper.

Good proof queries include:
- row counts
- sample rows
- simple aggregates / sanity checks

Deliverable:
- Fill in 3 queries below and show outputs.


In [ ]:

# TODO Query 1: Row counts for each table
#


In [ ]:

# TODO Query 2: Show the largest market cap stock in your quotes table



In [ ]:

# TODO Query 3: Show the most recent 5 price rows for your focus ticker





## Part 11 — Reflection (Required)

Write 3–6 sentences answering:
1. Which endpoint was easiest to work with, and why?
2. What was hardest part of this ETL pipeline?
3. If this ran daily in production, what would you automate or change?


In [ ]:

reflection = """
TODO
"""

print(reflection)



## Done

Before submitting:
- Run the notebook top-to-bottom (Restart & Run All)
- Confirm your proof queries show real rows
- Submit the Kaggle Notebook URL in Canvas
